# GANForge -- GANSynth (Standalone Notebook)

**Audio / music generation**

This is a **self-contained extract** of the GANSynth section from
the [GANForge Master Notebook](GANForge_Master_Notebook.ipynb) -- every
shared setup utility it needs (imports, device selection, seeding,
`weights_init`, `show_images`, checkpointing, etc.) is included below in
Section 0, so this notebook runs **top to bottom on its own**, in Colab
or locally, without needing any other file.

> **⚠️ Compute note.** See the Section 0 GPU warning below. Heavier
> architectures (Progressive GAN, StyleGAN2, BigGAN, StackGAN, GameGAN)
> include a lightweight from-scratch training loop sized for a single
> GPU (or CPU, slowly) plus, where realistically available, code to load
> strong pretrained weights for inference.

<a id="0"></a>
## 0. Global Setup & Utilities

Everything shared across all twelve GANs lives here: installs, imports,
device selection, seeding, and small helper functions (`weights_init`,
`show_images`, `save_checkpoint`, `Logger`, an `EMA` helper, and a spectral
-norm-aware conv wrapper). Re-using these keeps every later section short
and focused on what makes *that* GAN different.

**🚀 Pro Tip.** In Google Colab: `Runtime -> Change runtime type -> T4 GPU`
before running anything. Everything here auto-detects CUDA and degrades
gracefully to CPU (slow but functional) if none is found.

In [ ]:
# Colab-only: uncomment to install packages not preinstalled in Colab.
# All of these are optional -- every dataset loader below has an offline
# synthetic fallback, so the notebook still runs without them.
# !pip install -q torchaudio datasets soundfile pillow-simd torchmetrics
# !pip install -q pytorch-fid clean-fid


In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import io
import math
import time
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    # Minimal shim so the notebook still runs if tqdm is somehow missing.
    def tqdm(iterable, *args, **kwargs):
        return iterable

warnings.filterwarnings("ignore")  # GAN training is noisy (deprecation spam); keep output readable

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)


In [ ]:
# ---------------------------------------------------------------------------
# Device setup
# ---------------------------------------------------------------------------
# We centralize device selection so every model/tensor in the notebook is
# created consistently. All training loops below check DEVICE.type == "cuda"
# to decide whether to use mixed precision (torch.cuda.amp), which roughly
# halves memory use and often speeds up training 2-3x on modern GPUs.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found -- running on CPU. Reduce EPOCHS/BATCH_SIZE in "
          "every section for reasonable runtimes.")


In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
# GANs are notoriously sensitive to initialization -- fixing every RNG makes
# results reproducible across reruns, which matters a lot when you are
# debugging "why did my generator collapse this time".
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
# ---------------------------------------------------------------------------
# Shared helper functions used across every GAN section below
# ---------------------------------------------------------------------------

def weights_init(m: nn.Module):
    '''DCGAN-style weight init (Radford et al., 2015).

    Why: GAN training is a saddle-point optimization between two networks
    that constantly move each other's loss landscape. Starting both nets
    from small, zero-centered weights (N(0, 0.02)) avoids one network
    dominating early purely due to init scale, which is a common cause of
    early, unrecoverable mode collapse.

    Uses isinstance (NOT a substring match on the class name) because many
    custom container classes in this notebook -- PGAN_ConvBlock,
    ModulatedConv2d, StyleConvBlock -- have "Conv" in their own name; a
    naive `"Conv" in classname` check would incorrectly try to initialize
    THOSE containers directly (they have no `.weight` of their own) instead
    of only their actual nn.Conv2d/nn.Linear/nn.BatchNorm2d leaf children.
    '''
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and getattr(m, "affine", False):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def show_images(images: torch.Tensor, title: str = "", nrow: int = 8, figsize=(8, 8), normalize=True):
    '''Display a batch of image tensors (B, C, H, W) in [-1, 1] or [0, 1] as a grid.'''
    images = images.detach().cpu()
    grid = vutils.make_grid(images, nrow=nrow, normalize=normalize, value_range=(-1, 1) if normalize else None, padding=2)
    plt.figure(figsize=figsize)
    plt.axis("off")
    plt.title(title)
    npimg = grid.numpy()
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


def save_checkpoint(state: dict, path: str):
    '''Persist model/optimizer state so long training runs survive a Colab disconnect.'''
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path: str, map_location=None):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location or DEVICE)


class EMA:
    '''Exponential Moving Average of generator weights.

    Why: instantaneous generator weights oscillate a lot during adversarial
    training (they are chasing a moving discriminator). Averaging weights
    over time (as StyleGAN2 and BigGAN both do) gives a much smoother,
    higher-quality generator for *inference*, at zero extra training cost.
    '''
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


class AvgMeter:
    '''Running mean tracker for loss logging.'''
    def __init__(self):
        self.sum, self.n = 0.0, 0

    def update(self, val, k=1):
        self.sum += float(val) * k
        self.n += k

    @property
    def avg(self):
        return self.sum / max(self.n, 1)


def denorm(x: torch.Tensor) -> torch.Tensor:
    '''Map a tensor from [-1, 1] (tanh output range) back to [0, 1] for display/saving.'''
    return (x.clamp(-1, 1) + 1) / 2


CHECKPOINT_DIR = "checkpoints"
SAMPLES_DIR = "samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
print("Shared utilities ready.")


<a id="11"></a>
## 11. GANSynth -- Audio / music generation

### 11.1 Problem Statement & Why this GAN

Engel et al. (2019) asked whether GANs -- until then almost exclusively an
image tool -- could generate **musical audio** faster and at higher
fidelity than the autoregressive models (WaveNet, SampleRNN) dominant at
the time. Their key insight: don't make the GAN output a raw waveform
(a very long, temporally-fine-grained 1-D signal that convolutional GANs
handle poorly); instead output a **time-frequency spectrogram image**
(so all the image-GAN machinery from Sections 1-10 applies directly), then
convert that spectrogram back to audio. GANSynth's full pipeline also
predicts **instantaneous frequency** (the phase derivative) alongside
magnitude for higher-fidelity phase reconstruction; we simplify to
**log-magnitude spectrograms + Griffin-Lim phase reconstruction** to keep
the demo dependency-free and fast, while keeping the core idea --
*generate audio by generating an image of audio* -- fully intact. This is
the **audio/music generation** problem, conditioned on musical pitch.

### 11.2 Data Collection & Preparation

Real NSynth is a ~30GB download of individual instrument notes --
inappropriate for a guaranteed-offline demo. We instead **synthesize** a
dataset of musical notes via **additive synthesis** (random-weighted sine
harmonics = random timbre, at a controlled MIDI pitch) using
`torchaudio`'s standard signal-processing transforms for the actual
spectrogram pipeline -- a real, working small mel/spectrogram dataset, as
suggested for this section. **Swap this for real NSynth
(`tensorflow-datasets`'s `nsynth` config, or a Hugging Face NSynth mirror)
for musically realistic timbres.**

In [ ]:
import torchaudio

GANSYNTH_SR = 16000
GANSYNTH_DURATION = 0.5                       # seconds per note
GANSYNTH_N_FFT = 126
GANSYNTH_HOP = 125                            # must be <= n_fft for istft/Griffin-Lim; chosen with n_fft to land near 64 time frames
GANSYNTH_N_FREQ = GANSYNTH_N_FFT // 2 + 1     # 64 -- chosen so the spectrogram is a clean square "image"
GANSYNTH_N_FRAMES = 64
GANSYNTH_PITCH_MIN, GANSYNTH_PITCH_MAX = 48, 72   # 2 octaves of MIDI pitches (C3 - C5)
GANSYNTH_N_PITCHES = GANSYNTH_PITCH_MAX - GANSYNTH_PITCH_MIN + 1
GANSYNTH_BATCH_SIZE = 32
GANSYNTH_EPOCHS = 8
GANSYNTH_NZ = 64

spec_transform = torchaudio.transforms.Spectrogram(n_fft=GANSYNTH_N_FFT, hop_length=GANSYNTH_HOP, power=2.0)
amp_to_db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)
griffinlim = torchaudio.transforms.GriffinLim(n_fft=GANSYNTH_N_FFT, hop_length=GANSYNTH_HOP, power=2.0, n_iter=32)

def midi_to_freq(midi: int) -> float:
    return 440.0 * (2.0 ** ((midi - 69) / 12.0))

def synth_note(midi_pitch: int, rng, n_harmonics: int = 6) -> torch.Tensor:
    '''Additive synthesis: sum of sine harmonics at random relative
    amplitudes (= a random "timbre") at the requested pitch's fundamental
    frequency, with a simple exponential decay envelope (like a plucked
    string). This is what makes each pitch's training examples diverse in
    timbre, giving the GAN something nontrivial to model beyond a single
    fixed waveform per pitch.'''
    freq = midi_to_freq(midi_pitch)
    t = np.linspace(0, GANSYNTH_DURATION, int(GANSYNTH_SR * GANSYNTH_DURATION), endpoint=False)
    harmonic_amps = rng.dirichlet(np.ones(n_harmonics))   # random mixture summing to 1 -> random timbre
    wave = np.zeros_like(t)
    for h in range(1, n_harmonics + 1):
        wave += harmonic_amps[h - 1] * np.sin(2 * np.pi * freq * h * t)
    envelope = np.exp(-3.0 * t / GANSYNTH_DURATION)
    wave = wave * envelope
    wave = wave / (np.max(np.abs(wave)) + 1e-8)
    wave = wave + rng.normal(0, 0.005, size=wave.shape)   # light noise floor, like real recordings
    return torch.tensor(wave, dtype=torch.float32)

def wave_to_spec_db(wave: torch.Tensor) -> torch.Tensor:
    spec_db = amp_to_db(spec_transform(wave.unsqueeze(0))).squeeze(0)   # (freq, time), roughly in [-80, 0] dB
    if spec_db.size(1) < GANSYNTH_N_FRAMES:
        spec_db = F.pad(spec_db, (0, GANSYNTH_N_FRAMES - spec_db.size(1)), value=-80.0)
    return spec_db[:, :GANSYNTH_N_FRAMES]

def spec_db_to_wave(spec_db: torch.Tensor) -> torch.Tensor:
    '''Approximate inverse: dB -> power spectrogram -> Griffin-Lim phase
    reconstruction. This is a classical DSP algorithm (no learned phase
    model), which is why real GANSynth's instantaneous-frequency approach
    sounds meaningfully cleaner -- but Griffin-Lim needs no extra network
    or training and is "good enough" to hear pitch/timbre clearly.'''
    spec_power = torch.pow(10.0, spec_db.clamp(-80, 0) / 10.0)
    return griffinlim(spec_power.unsqueeze(0)).squeeze(0)

class GANSynthDataset(Dataset):
    def __init__(self, n_per_pitch=24, seed=SEED):
        self.rng = np.random.default_rng(seed)
        self.pitch_idx_list = [p for p in range(GANSYNTH_N_PITCHES) for _ in range(n_per_pitch)]

    def __len__(self):
        return len(self.pitch_idx_list)

    def __getitem__(self, i):
        pitch_idx = self.pitch_idx_list[i]
        wave = synth_note(GANSYNTH_PITCH_MIN + pitch_idx, self.rng)
        spec_db = wave_to_spec_db(wave)
        spec_norm = (spec_db.clamp(-80, 0) + 40) / 40   # map [-80, 0] dB -> [-1, 1] (matches Tanh generator output)
        return spec_norm.unsqueeze(0), pitch_idx

gansynth_dataset = GANSynthDataset()
gansynth_loader = DataLoader(gansynth_dataset, batch_size=GANSYNTH_BATCH_SIZE, shuffle=True, drop_last=True)
spec_batch, pitch_batch = next(iter(gansynth_loader))
print(f"Synthetic GANSynth dataset: {len(gansynth_dataset)} notes across {GANSYNTH_N_PITCHES} pitches, "
      f"spectrogram shape {tuple(spec_batch.shape[-2:])}")
show_images(spec_batch[:16], title="Real training spectrograms (as images)", nrow=4, normalize=True)

try:
    from IPython.display import Audio, display
    display(Audio(spec_db_to_wave(spec_batch[0, 0] * 40 - 40).numpy(), rate=GANSYNTH_SR))
except Exception as e:
    print(f"Inline audio playback unavailable in this environment ({e}); "
          f"spec_db_to_wave(...) still returns a valid waveform tensor you can save with torchaudio.save(...).")


### 11.3 Model Architecture

In [ ]:
class GANSynth_Generator(nn.Module):
    '''Identical mechanism to Section 2's cGAN generator -- pitch-conditional
    image generation -- pointed at a spectrogram "image" instead of a
    photo. This is the central lesson of GANSynth: once audio is
    represented as a 2-D time-frequency image, it IS an image-generation
    problem.'''
    def __init__(self, nz=GANSYNTH_NZ, n_pitches=GANSYNTH_N_PITCHES, ngf=64, embed_dim=32):
        super().__init__()
        self.pitch_embed = nn.Embedding(n_pitches, embed_dim)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz + embed_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8), nn.ReLU(True),                                    # 1 -> 4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),                                    # 4 -> 8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),                                    # 8 -> 16
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),                                        # 16 -> 32
            nn.ConvTranspose2d(ngf, 1, 4, 2, 1, bias=False), nn.Tanh(),                # 32 -> 64
        )

    def forward(self, z, pitch):
        pe = self.pitch_embed(pitch)
        zc = torch.cat([z, pe], dim=1)
        return self.net(zc.view(zc.size(0), -1, 1, 1))


class GANSynth_Discriminator(nn.Module):
    def __init__(self, n_pitches=GANSYNTH_N_PITCHES, ndf=64, spec_size=GANSYNTH_N_FRAMES):
        super().__init__()
        self.spec_size = spec_size
        self.pitch_embed = nn.Embedding(n_pitches, spec_size * spec_size)
        self.net = nn.Sequential(
            nn.Conv2d(2, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf * 8), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
        )

    def forward(self, spec, pitch):
        pitch_map = self.pitch_embed(pitch).view(-1, 1, self.spec_size, self.spec_size)
        return self.net(torch.cat([spec, pitch_map], dim=1)).view(-1)


G_gansynth = GANSynth_Generator().to(DEVICE).apply(weights_init)
D_gansynth = GANSynth_Discriminator().to(DEVICE).apply(weights_init)


### 11.4 Loss Functions & Optimizers

In [ ]:
gansynth_criterion = nn.BCEWithLogitsLoss()
gansynth_opt_G = torch.optim.Adam(G_gansynth.parameters(), lr=2e-4, betas=(0.5, 0.999))
gansynth_opt_D = torch.optim.Adam(D_gansynth.parameters(), lr=2e-4, betas=(0.5, 0.999))


### 11.5 Training Loop

In [ ]:
gansynth_history = []
for epoch in range(GANSYNTH_EPOCHS):
    meters = {"G": AvgMeter(), "D": AvgMeter()}
    pbar = tqdm(gansynth_loader, desc=f"[GANSynth] epoch {epoch+1}/{GANSYNTH_EPOCHS}")
    for spec, pitch in pbar:
        spec, pitch = spec.to(DEVICE), pitch.to(DEVICE)
        b = spec.size(0)

        D_gansynth.zero_grad(set_to_none=True)
        logits_real = D_gansynth(spec, pitch)
        loss_d_real = gansynth_criterion(logits_real, torch.ones(b, device=DEVICE))
        z = torch.randn(b, GANSYNTH_NZ, device=DEVICE)
        fake_pitch = torch.randint(0, GANSYNTH_N_PITCHES, (b,), device=DEVICE)
        fake_spec = G_gansynth(z, fake_pitch)
        logits_fake = D_gansynth(fake_spec.detach(), fake_pitch)
        loss_d_fake = gansynth_criterion(logits_fake, torch.zeros(b, device=DEVICE))
        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        gansynth_opt_D.step()

        G_gansynth.zero_grad(set_to_none=True)
        logits_fake_g = D_gansynth(fake_spec, fake_pitch)
        loss_g = gansynth_criterion(logits_fake_g, torch.ones(b, device=DEVICE))
        loss_g.backward()
        gansynth_opt_G.step()

        meters["D"].update(loss_d.item()); meters["G"].update(loss_g.item())
        pbar.set_postfix(loss_D=f"{loss_d.item():.3f}", loss_G=f"{loss_g.item():.3f}")

    gansynth_history.append((meters["G"].avg, meters["D"].avg))
    with torch.no_grad():
        G_gansynth.eval()
        demo_pitch = torch.arange(GANSYNTH_N_PITCHES, device=DEVICE)[:16] % GANSYNTH_N_PITCHES
        demo_z = torch.randn(16, GANSYNTH_NZ, device=DEVICE)
        sample = G_gansynth(demo_z, demo_pitch)
        G_gansynth.train()
    show_images(sample, title=f"GANSynth generated spectrograms after epoch {epoch+1}", nrow=4)
    save_checkpoint({"G": G_gansynth.state_dict(), "D": D_gansynth.state_dict()}, f"{CHECKPOINT_DIR}/gansynth_epoch{epoch+1}.pt")

print("GANSynth training complete.")


### 11.6 Inference / Generation Function

In [ ]:
@torch.no_grad()
def gansynth_generate(midi_pitch: int, seed: int = None) -> torch.Tensor:
    '''Text/number-in (a MIDI pitch), audio waveform out -- the clean
    end-to-end user-facing entry point for this section.'''
    if seed is not None:
        torch.manual_seed(seed)
    pitch_idx = max(0, min(GANSYNTH_N_PITCHES - 1, midi_pitch - GANSYNTH_PITCH_MIN))
    G_gansynth.eval()
    z = torch.randn(1, GANSYNTH_NZ, device=DEVICE)
    pitch_t = torch.tensor([pitch_idx], device=DEVICE)
    spec_norm = G_gansynth(z, pitch_t)[0, 0].cpu()
    G_gansynth.train()
    spec_db = spec_norm * 40 - 40
    return spec_db_to_wave(spec_db)

wave = gansynth_generate(midi_pitch=60, seed=0)   # 60 = middle C
print(f"Generated {wave.numel()/GANSYNTH_SR:.1f}s of audio at {GANSYNTH_SR}Hz for MIDI pitch 60 (middle C).")
try:
    from IPython.display import Audio, display
    display(Audio(wave.numpy(), rate=GANSYNTH_SR))
except Exception:
    pass


### 11.7 Visualization & Results

In [ ]:
gs, ds = zip(*gansynth_history)
plt.figure(figsize=(6, 4)); plt.plot(gs, label="G"); plt.plot(ds, label="D")
plt.legend(); plt.title("GANSynth training losses"); plt.show()

# Pitch sweep: same z (same "timbre"), varying requested pitch -- checks
# that pitch conditioning actually shifts the spectrogram's energy to a
# higher fundamental frequency band, not just an arbitrary different image.
with torch.no_grad():
    G_gansynth.eval()
    z_fixed = torch.randn(1, GANSYNTH_NZ, device=DEVICE).repeat(6, 1)
    pitches_to_show = torch.linspace(0, GANSYNTH_N_PITCHES - 1, 6).long().to(DEVICE)
    specs = G_gansynth(z_fixed, pitches_to_show)
    G_gansynth.train()
show_images(specs, title="Same latent, increasing pitch (low -> high, left to right)", nrow=6, figsize=(12, 2))

# Latent interpolation in timbre space at a FIXED pitch -- audible/visible
# smooth timbre morphing is the audio analogue of Section 1's image
# interpolation sanity check.
with torch.no_grad():
    G_gansynth.eval()
    z1, z2 = torch.randn(1, GANSYNTH_NZ, device=DEVICE), torch.randn(1, GANSYNTH_NZ, device=DEVICE)
    alphas = torch.linspace(0, 1, 6, device=DEVICE).view(-1, 1)
    z_interp = (1 - alphas) * z1 + alphas * z2
    pitch_fixed = torch.full((6,), GANSYNTH_N_PITCHES // 2, dtype=torch.long, device=DEVICE)
    specs_interp = G_gansynth(z_interp, pitch_fixed)
    G_gansynth.train()
show_images(specs_interp, title="Timbre interpolation at fixed pitch", nrow=6, figsize=(12, 2))


### 11.8 User Input → Output Demo

In [ ]:
USER_MIDI_PITCH = 67    # 67 = G4
RANDOM_SEED = 3

user_wave = gansynth_generate(USER_MIDI_PITCH, seed=RANDOM_SEED)
print(f"User requested MIDI pitch {USER_MIDI_PITCH} (~{midi_to_freq(USER_MIDI_PITCH):.1f} Hz)")
try:
    from IPython.display import Audio, display
    display(Audio(user_wave.numpy(), rate=GANSYNTH_SR))
except Exception:
    pass
plt.figure(figsize=(6, 2)); plt.plot(user_wave.numpy()); plt.title("Generated waveform"); plt.show()


### 11.9 Key Insights & Limitations of this GAN

- 💡 **Insight.** "Turn X into an image, then use image-GAN machinery" is
  a broadly reusable trick beyond audio (e.g. time-series-as-image,
  graph-layouts-as-image) whenever X has a natural, roughly
  translation-invariant 2-D structure.
- ⚠️ **Warning.** Griffin-Lim phase reconstruction is the main fidelity
  bottleneck here, not the GAN -- it produces audible "phasiness"/metallic
  artifacts that a learned phase model (real GANSynth's instantaneous
  -frequency channel, or a neural vocoder like WaveGlow/HiFi-GAN) would
  remove.
- ⚠️ **Warning.** Our additive-synthesis training data has much less
  timbral realism than real recorded instruments (NSynth) -- pitch
  conditioning transfers directly to real data, but expect the model to
  need more capacity/data to capture real instrument timbre variation.

**This GAN solved the problem of: audio/music generation, conditioned on
a requested pitch, by reframing "generate a sound" as "generate a
spectrogram image" -- reusing conditional image-GAN machinery for a
non-image modality.**